# Do the V columns fall into families that share a missingness pattern?

> **Provenance.** The idea that the V block partitions by missingness comes from Chris Deotte, 'EDA for Columns V and ID'. See ATTRIBUTION.md.

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [11]:
import polars as pl
from IPython.display import Markdown, display

df_txn = pl.scan_csv("../../kaggle/raw/train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
df_id = pl.scan_csv("../../kaggle/raw/train_identity.csv", infer_schema_length=10000, null_values=[""]).collect()
df = df_txn.join(df_id, on="TransactionID", how="left")

In [14]:
families = {
    "`V1-V339` (anonymized)": [c for c in df.columns if c.startswith("V")],
    "`id_01-id_38` (identity)": [c for c in df.columns if c.startswith("id_")],
    "`D1-D15` (time deltas)": [c for c in df.columns if c.startswith("D") and c != "DeviceType" and c != "DeviceInfo"],
    "`C1-C14` (counters)": [c for c in df.columns if c.startswith("C")],
    "`M1-M9` (match flags)": [c for c in df.columns if c.startswith("M")],
    "`card1-card6`": [c for c in df.columns if c.startswith("card")],
    "`addr` / `dist`": [c for c in df.columns if c.startswith(("addr", "dist"))],
    "Email domains": [c for c in df.columns if "emaildomain" in c]
}

data = []
for name, cols in families.items():
    if not cols:
        continue
    rates = [df.select(pl.col(c).null_count()).item() / df.height for c in cols]
    data.append({
        "Family": name,
        "Columns": len(cols),
        "Min null rate": f"{min(rates)*100:.1f}%",
        "Median": f"{pl.Series(rates).median()*100:.1f}%",
        "Max": f"{max(rates)*100:.1f}%"
    })

df_fam = pl.DataFrame(data)
display(Markdown(df_fam.to_pandas().to_markdown(index=False)))

| Family                   |   Columns | Min null rate   | Median   | Max   |
|:-------------------------|----------:|:----------------|:---------|:------|
| `V1-V339` (anonymized)   |       339 | 0.0%            | 47.3%    | 86.1% |
| `id_01-id_38` (identity) |        38 | 75.6%           | 82.4%    | 99.2% |
| `D1-D15` (time deltas)   |        15 | 0.2%            | 52.5%    | 93.4% |
| `C1-C14` (counters)      |        14 | 0.0%            | 0.0%     | 0.0%  |
| `M1-M9` (match flags)    |         9 | 28.7%           | 47.7%    | 59.3% |
| `card1-card6`            |         6 | 0.0%            | 0.3%     | 1.5%  |
| `addr` / `dist`          |         4 | 11.1%           | 35.4%    | 93.6% |
| Email domains            |         2 | 16.0%           | 46.4%    | 76.8% |

## V-Block NaN Clustering

Grouping the 339 anonymized columns by their number of missing values:


In [15]:
# Calculate null rate for each V column
v_cols = [c for c in df.columns if c.startswith("V")]
null_rates = [(c, df.select(pl.col(c).null_count()).item() / df.height) for c in v_cols]

# Group by null rate to find families
df_nulls = pl.DataFrame(null_rates, schema=["column", "null_rate"], orient="row")
df_grouped = (
    df_nulls.group_by("null_rate")
    .agg([
        pl.len().alias("Columns in group"),
        pl.col("column").alias("cols")
    ])
    .sort("Columns in group", descending=True)
)

# Format example range (min and max V col)
df_final = df_grouped.with_columns([
    pl.col("null_rate").map_elements(lambda x: f"{x*100:.1f}%", return_dtype=pl.String).alias("Null rate"),
    pl.col("cols").map_elements(
        lambda cols: f"V{min(int(c[1:]) for c in cols)}-V{max(int(c[1:]) for c in cols)}", return_dtype=pl.String
    ).alias("Example range")
]).select(["Columns in group", "Null rate", "Example range"])

display(Markdown(df_final.to_pandas().to_markdown(index=False)))

|   Columns in group | Null rate   | Example range   |
|-------------------:|:------------|:----------------|
|                 46 | 77.9%       | V217-V278       |
|                 43 | 0.1%        | V95-V137        |
|                 32 | 0.0%        | V279-V321       |
|                 31 | 76.4%       | V167-V216       |
|                 23 | 12.9%       | V12-V34         |
|                 22 | 13.1%       | V53-V74         |
|                 20 | 15.1%       | V75-V94         |
|                 19 | 76.3%       | V169-V210       |
|                 18 | 28.6%       | V35-V52         |
|                 18 | 86.1%       | V322-V339       |
|                 18 | 86.1%       | V138-V163       |
|                 16 | 76.1%       | V220-V272       |
|                 11 | 0.2%        | V281-V315       |
|                 11 | 86.1%       | V143-V166       |
|                 11 | 47.3%       | V1-V11          |

15 distinct groups, and not one singleton. Every V column shares its missingness pattern with at least ten others, and the groups map onto contiguous index ranges — the signature of columns generated together, from the same source, at the same stage.

This is the empirical basis for the reduction strategy: group by missingness pattern first, then prune within each group by correlation. 

It also explains why a variance threshold was the wrong instrument — variance says nothing about membership of a block of 46 columns that appear and disappear together.


# Redundancy

**Which columns carry the same signal as their neighbours?**

The `V` block is 339 anonymized columns generated in families: contiguous indices sharing a
missingness pattern, heavily correlated inside each family. Keeping all of them costs
training time and interpretability without buying accuracy.

Code: [`src/fraud_detection/techniques/redundancy.py`](../../src/fraud_detection/techniques/redundancy.py)

## Three steps, only two of them mechanical

1. **Group by missingness.** Columns produced together are absent together, so the null
   count partitions the block. Mechanical.
2. **Split each family into correlated sub-groups.** Judgement, read off correlation
   heatmaps by a human. **Taken as given** from
   [`references/column-groups-v.json`](../../references/column-groups-v.json), not
   re-derived.
3. **Keep one representative per sub-group** — the column with the most distinct values.
   Mechanical.

Step 2 being somebody's judgement is exactly why step 4 exists.

## Step 1 confirms itself

Grouping the 339 columns by exact null count yields **15 families**, sized
46, 43, 32, 31, 23, 22, 20, 19, 18, 18, 18, 16, 11, 11, 11.

That is the same 15 the [EDA](../README.md) §5 arrived at from the other direction. Two independent routes, one answer.

## The pinned partition has a hole

It covers **338 of 339** columns. `V155` appears in no group — it fell between two hand-drawn partitions of the `V138-V166` family.

The code adds it back as a group of one, because **silence is not a decision**: a reduction has to rule on every member of the block, and a column nobody assigned is not a column to drop. That is why the reduction keeps **129** representatives, not the 128 the source notebook lists.

## Result

|                      |         |
| -------------------- | ------: |
| Columns in           |     339 |
| Representatives kept | **129** |
| Dropped              |     210 |
| Reduction            | **62%** |

## Step 4: does the partition hold?

For every sub-group, compare the **weakest correlation inside it** against the
**strongest correlation to a column outside it but inside the same family**. A sub-group
that means anything should be more tightly bound internally than to its neighbours.

|                                                                     |           |
| ------------------------------------------------------------------- | --------: |
| Groups                                                              |       128 |
| Measurable (more than one column, and something to compare against) |        93 |
| Hold                                                                |    **73** |
| Do not hold                                                         |    **20** |
| Share holding                                                       | **78.5%** |

The 35 unmeasurable groups are single columns: there is no internal correlation to compute,
so the verdict is unknown rather than a pass.

Worst offenders — the weakest link inside the group is far weaker than its strongest tie
outside it:

| Group                  | Size | min inside | max outside |
| ---------------------- | ---: | ---------: | ----------: |
| `V108+V109+V110+V114`  |    4 |      0.455 |       0.803 |
| `V186+V187+V190…+V199` |    8 |      0.595 |       0.892 |
| `V214+V215+V216`       |    3 |      0.591 |       0.849 |
| `V263+V265+V264`       |    3 |      0.636 |       0.893 |

A partition drawn by eye off heatmaps on a 20% sample fails its own property in one group
in five. That does not invalidate the reduction — 62% fewer columns is still 62% fewer
columns — but it says what the _group boundary_ is worth, and the number travels into the
contract as `qualification` so a reader of `feature_contract.json` can see it.

Full output: `references/v-block-audit.csv`.

## The block splits in two, and that is measurable

The source notebook records an observation in passing: the first hundred `V` columns do not
correlate much with the remaining 239, while inside each half the correlation crosses
missingness-group boundaries. Checked here on a 120k sample:

|                               | Median \|r\| | Share above 0.5 |
| ----------------------------- | -----------: | --------------: |
| `V1-V100` against `V101-V339` |    **0.033** |        **2.4%** |
| within `V1-V100`              |        0.095 |           15.1% |
| within `V101-V339`            |        0.060 |           15.5% |

It holds. The two halves are essentially unrelated — 2.4% of cross pairs reach 0.5 against
roughly 15% inside either half — which says the `V` block is two independently generated
families of columns rather than one.

That matters for reduction: a correlation-based method can safely be run on each half
separately, and any group spanning the boundary would be an artefact rather than a family.
None in the pinned partition does.

## Cross-check

[PCA on the same groups](selection.md) independently flags `V108+V109+V110+V114` as the
worst — its first component retains 75.1% of the group's variance against a median of 94.9%
elsewhere. Two unrelated methods, same group. That is the strongest evidence available that
the failure is in the partition rather than in the audit.
